# Funkcije višjega reda

V Pythonu so funkcije objekti, ki jih lahko priredimo spremenljivkam, podamo kot argumente drugim funkcijam ali vrnemo kot rezultate funkcij. Funkcijam, ki sprejmejo funkcijo kot argument ali vrnejo funkcijo kot rezultat, pravimo **funkcije višjega reda** (angl. _higher-order functions_).

## Funkciji `map` in `filter`

Primera funkcij višjega reda sta funkciji `map` in `filter`: obe kot prvi argument sprejmeta funkcijo. Z izpeljanimi seznami lahko napišemo različici, ki rezultate zbereta v seznam:

  * `map_is(f, vir)` uporabi funkcijo `f` na vsakem elementu vira;
  * `filter_is(f, vir)` izbere elemente, za katere ima rezultat klica `f` logično vrednost `True`.

To sta poenostavljeni različici: vgrajeni funkciji `map` in `filter` vrneta iteratorja in rezultate ustvarjata sproti, naši funkciji pa takoj ustvarita seznama. Poleg tega vgrajeni `map` lahko sprejme več virov, `filter` pa kot prvi argument tudi `None`.

In [1]:
def map_is(f, vir):
    return [f(e) for e in vir]


def filter_is(f, vir):
    return [e for e in vir if f(e)]

Funkciji `f`, ki jo podamo funkciji `filter_is`, pogosto pravimo **predikat**: običajno vrne logično vrednost `True` ali `False`. Vendar to ni obvezno, saj Python v pogoju preveri logično vrednost poljubnega objekta. Na primer, število `0` in prazen niz imata logično vrednost `False`, neničelno število in neprazen niz pa `True`.

Preverimo, ali funkciji delujeta po pričakovanjih. Ustvarimo seznam kvadratov celih števil od 0 do 9, izberemo liha števila in nato izračunamo še njihove kvadrate:

In [2]:
print(map_is(lambda e: e ** 2, range(10)))
print(filter_is(lambda e: e % 2 == 1, range(10)))
print(map_is(lambda e: e ** 2, filter_is(lambda e: e % 2 == 1, range(10))))

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
[1, 3, 5, 7, 9]
[1, 9, 25, 49, 81]


## Vaja: združevanje elementov

Definiraj funkcijo `reduce(f, e, s)`, ki sprejme funkcijo `f`, začetno vrednost `e` in seznam `s`. Funkcija `f` sprejme dva argumenta in ju združi v novo vrednost. Funkcija `reduce` naj začne z vrednostjo `e`, nato pa za vsak element seznama, od leve proti desni, trenutni rezultat nadomesti z rezultatom klica `f(trenutni_rezultat, element)`. Na koncu naj vrne dobljeno vrednost; za prazen seznam naj vrne `e`.

Za seznam treh elementov tako izračuna `f(f(f(e, s[0]), s[1]), s[2])`. Preveri vsaj naslednje primere:

  * `reduce(lambda a, b: a + b, 0, [1, 2, 3])` naj vrne `6`;
  * `reduce(lambda a, b: a * b, 1, [2, 3, 4])` naj vrne `24`;
  * `reduce(lambda a, b: a + b, 0, [])` naj vrne `0`.

Pri seštevanju je `0` **nevtralni element** (ali enota), pri množenju pa `1`. Nevtralni element `e` za operacijo `f` izpolnjuje `f(a, e) = f(e, a) = a`. Začetna vrednost pri zaporednem združevanju ni nujno nevtralni element, je pa taka izbira pogosto uporabna.

Premisli, katere lastnosti operacije zagotavljajo neodvisnost rezultata od razvrstitve oklepajev in od vrstnega reda elementov. Pri določeni smeri združevanja asociativnost ni potrebna, mora pa funkcija `f` sprejeti vsak vmesni rezultat in naslednji element. Za neodvisnost od razvrstitve oklepajev potrebujemo asociativnost, za poljubno preurejanje elementov pa ob asociativnosti še komutativnost. Upoštevaj, da seštevanje števil s plavajočo vejico zaradi zaokroževanja ni povsem asociativno.

Primer asociativne operacije, ki ni komutativna, je združevanje nizov: `("a" + "b") + "c"` in `"a" + ("b" + "c")` oba vrneta `"abc"`, preurejanje nizov v vrstni red `"b", "a", "c"` pa da rezultat `"bac"`.

Python ponuja tudi [funkcijo `functools.reduce`](https://docs.python.org/3/library/functools.html#functools.reduce), ki ima drugačen vrstni red argumentov: `reduce(f, s, e)`.

## Funkcija kot argument

Funkcije višjega reda omogočajo pisanje splošnih algoritmov, katerih delovanje prilagodimo s podano funkcijo. Primer je numerično računanje določenega integrala

$$\int_a^b f(x)\,\mathrm{d}x.$$

S sestavljeno trapezno formulo izračunamo približek integrala funkcije $f$ na intervalu $[a, b]$. Interval razdelimo na $n$ enako dolgih podintervalov s krajišči $x_0 = a, \ldots, x_n = b$ in dolžino $\Delta x = (b-a)/n$. Približek je

$$\int_a^b f(x)\,\mathrm{d}x \approx \sum_{i=0}^{n-1} \frac{f(x_i) + f(x_{i+1})}{2}\,\Delta x.$$

V spodnji funkciji parameter `h` določa največjo dolžino podintervala. Število podintervalov izberemo z zaokrožitvijo navzgor, dejanski korak pa prilagodimo tako, da pokrijemo celoten interval. V tej različici predpostavljamo končni meji `a < b` ter pozitiven in končen `h`:

In [3]:
from math import ceil, isfinite, pi, sin


def dolocen_integral(f, interval, h=0.01):
    a, b = interval
    if not (isfinite(a) and isfinite(b) and a < b):
        raise ValueError("Meji morata biti končni in spodnja meja manjša od zgornje.")
    if not isfinite(h) or h <= 0:
        raise ValueError("Korak h mora biti pozitiven in končen.")

    n = ceil((b - a) / h)
    korak = (b - a) / n
    tocke = [a + korak * i for i in range(n)] + [b]
    vrednosti = [f(x) for x in tocke]
    prispevki = [
        (vrednosti[i] + vrednosti[i + 1]) * korak / 2
        for i in range(n)
    ]
    return sum(prispevki)


print(dolocen_integral(sin, (0, pi)))  # Točna vrednost integrala je 2.
print(dolocen_integral(lambda x: 1, (0, 1), h=0.3))  # Točna vrednost je 1.

1.9999834221537534
1.0


Funkcija iz para `interval` prebere meji `a` in `b` ter preveri meji in korak. Nato določi število podintervalov `n` in dejansko dolžino podintervala `korak`. Seznam `tocke` vsebuje obe krajišči intervala, zato ne izpustimo njegovega konca niti takrat, ko dolžina intervala ni večkratnik `h`.

Funkcijo `f` izračunamo enkrat v vsaki točki, dobljene vrednosti pa uporabimo za izračun prispevkov trapezov. Prispevki pod osjo $x$ so negativni: računamo približek določenega integrala, ki upošteva predznak, in ne skupne geometrijske ploščine.

Funkcijo `dolocen_integral` lahko uporabimo za različne funkcije `f`, ki so definirane in imajo končne realne vrednosti na obravnavanem intervalu. Za dovolj gladke funkcije manjši korak praviloma izboljša približek, vendar zahteva več izračunov. Pri funkcijah z nezveznostmi ali singularnostmi je lahko potrebna drugačna numerična metoda.

Za vajo primerjaj približke integrala funkcije `sin` na intervalu od `0` do `pi` pri korakih `0.5`, `0.1` in `0.01` s točno vrednostjo `2`.

Pri podatkovni analizi bomo uporabljali predvsem funkcije višjega reda, ki sprejmejo funkcijo kot argument. V nadaljevanju si oglejmo še funkcije, ki vrnejo funkcijo kot rezultat. To je napredna tema, ki je ne bomo potrebovali za nadaljnje primere v tem semestru.

## Napredna tema: funkcija kot rezultat

Spomnimo se definicije kompozituma $f \circ g$: $(f \circ g)(x) = f(g(x))$. Za funkcijo $f\colon X \to X$ lahko definiramo ponavljajoče se sestavljanje:

  * $f^0(x) = x$ (identiteta);
  * $f^1(x) = f(x)$;
  * $f^n(x) = f(f^{n-1}(x))$ za celo število $n \geq 1$.

Zapis $f^n$ tukaj pomeni $n$-kratno sestavljanje funkcije, ne potenciranja vrednosti $f(x)$. Predpostavka $f\colon X \to X$ zagotavlja, da lahko rezultat enega klica uporabimo kot argument naslednjega.

Definirajmo funkcijo `fn`, ki sprejme funkcijo `f` in nenegativno celo število `n` ter **vrne novo funkcijo**, ki na argumentu `x` izračuna $f^n(x)$:

In [4]:
def fn(f, n):
    if not isinstance(n, int):
        raise TypeError("Število ponovitev n mora biti celo število.")
    if n < 0:
        raise ValueError("Število ponovitev n mora biti nenegativno.")

    def kompozitum(x):
        rezultat = x
        for _ in range(n):
            rezultat = f(rezultat)
        return rezultat

    return kompozitum


def povecaj(x):
    return x + 1


povecaj_desetkrat = fn(povecaj, 10)
print(type(povecaj_desetkrat))
print(povecaj_desetkrat(0))
print(fn(povecaj, 0)(7))

<class 'function'>
10
7


V funkciji `fn` najprej preverimo `n`, nato definiramo notranjo funkcijo `kompozitum` in jo vrnemo kot rezultat. Notranja funkcija pri posameznem klicu začne z vrednostjo `x` in funkcijo `f` uporabi natanko `n`-krat. Če je `n = 0`, se zanka ne izvede in funkcija vrne `x`.

Notranja funkcija ima dostop do argumentov `f` in `n` zunanje funkcije tudi po tem, ko se klic `fn` konča. Funkciji skupaj z dostopom do spremenljivk iz zunanjega dosega pravimo **zaprtje** (angl. _closure_).

Klic `fn(povecaj, 10)` torej vrne funkcijo, ki jo shranimo v spremenljivko `povecaj_desetkrat`. Izpis njenega tipa pokaže `<class 'function'>`, klic `povecaj_desetkrat(0)` pa vrne `10`. Isto lahko zapišemo kot `fn(povecaj, 10)(0)`: prvi par oklepajev ustvari funkcijo, drugi pa jo pokliče. Zadnji primer, `fn(povecaj, 0)(7)`, vrne `7`, ker funkcije `povecaj` ne uporabimo niti enkrat.

Za vajo primerjaj `fn(lambda x: 2 * x, 3)(2)` in `(2 * 2) ** 3` ter pojasni razliko med ponavljajočim se sestavljanjem in potenciranjem.